<a href="https://colab.research.google.com/github/ngtupolyakov/Lab3/blob/main/Lab3WebScr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
url = "https://ru.wikipedia.org/wiki/Ветроэнергетика"

headers = {
    "User-Agent": "Chrome"
}

response = requests.get(url, headers=headers)

tables = pd.read_html(response.text, index_col = 0)

id_table = 2
df_power = tables[id_table]

df_power

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt

url = "https://ru.wikipedia.org/wiki/Ветроэнергетика"
headers = {"User-Agent": "Chrome"}
response = requests.get(url, headers=headers)
tables = pd.read_html(response.text, index_col=0)
df_power = tables[2]


years = ['1985', '1990', '2000', '2010', '2015', '2016', '2017', '2018', '2019', '2020']
df_power[years] = df_power[years].apply(pd.to_numeric, errors='coerce')
df_power[years] = df_power[years]/10
df_power[years] = df_power[years].fillna(0)
df_power['Сумма'] = df_power[years].sum(axis=1, numeric_only=True)
df_grouped = df_power.groupby(df_power.index).sum()

yearsNum = [int(i) for i in years]
country_col = df_power.columns[0]
df_power = df_power.set_index(country_col)
df_grouped = df_power.groupby(df_power.index).sum()


plt.figure(figsize=(16, 12))
for country in df_grouped.index:
    plt.plot(yearsNum, df_grouped.loc[country, years].values, label=country, marker='.')
    y_values = df_grouped.loc[country, years].values
    plt.plot(yearsNum, y_values, label=country)
    last_x = yearsNum[-1]
    last_y = y_values[-1]
    if last_y > 30:
        plt.text(last_x + 0.2, last_y, country, fontsize=5, va='center')
plt.title('Ветроэнергетика по всем странам')
plt.xlabel('Год')
plt.ylabel('Производство, ТВт⋅ч')
plt.grid()
plt.show()

df_power


In [ ]:
df_countries_sum = df_grouped['Сумма'].sort_values(ascending=False)

axes = df_countries_sum.head(75).plot(kind='bar', figsize=(20, 6), color='green')

for p in axes.patches:
    val = round(p.get_height(), 1)
    if val > 0:
        axes.annotate(str(val), (p.get_x(), p.get_height() * 1.01))
plt.title('Общая выработка, ТВт⋅ч')
plt.xticks()
axes.grid(True)
plt.tight_layout()
plt.show()

In [ ]:

df_2000 = df_grouped['2000'].sort_values(ascending=False)

df_pie = df_2000.head(9).copy()
df_pie['Другие'] = df_2000.iloc[9:].sum()

plt.figure(figsize=(8, 8))
plt.pie(df_pie, labels=df_pie.index)
plt.title('Доли стран в ветроэнергетике (2000 год)', fontweight='bold')
plt.show()

In [ ]:

from bokeh.plotting import figure, output_file, show
from bokeh.io import output_notebook
import numpy as np

p = figure(width=800, height=400, title="Развитие ветроэнергетики по странам")
colors = ['magenta', 'red', 'yellow', 'green', 'cyan', 'blue', 'black']
top_countries = list(df_grouped['Сумма'].sort_values(ascending=False).head(7).index)
x_years = [int(y) for y in years]
for i, country in enumerate(top_countries):
    values = df_grouped.loc[country, years].values
    p.line(x_years, values,
           color=colors[i % len(colors)],
           legend_label=str(country),
           line_width=3)
    p.scatter(x_years, values,
              color=colors[i % len(colors)],
              alpha=0.5, size=8)
# Настройка оси Y для информативности
p.yaxis.axis_label = "Производство, ТВт⋅ч"
p.xaxis.axis_label = "Год"
show(p)